# Feature Engineering: Metadata & Interactions

Creates metadata features, interaction terms, and category-relative features from existing columns.

**Run after:** `04_advanced_nlp.ipynb`

**New features (~15):**
- Price: log_price, price_bin, price_vs_category_median
- Completeness: has_multiple_images, num_colors_available
- Interactions: title_length_x_sharpness, price_x_image_count, text_richness_x_image_count
- Category-relative: title_length_vs_category_median

In [1]:
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')

DATA_PATH = "../../data/products_with_image_feats.csv"

print("Loading dataset...")
df = pd.read_csv(DATA_PATH)
print(f"Loaded {len(df)} products, {len(df.columns)} columns")

# Check available columns for engineering
for col in ['price', 'sd_price', 'sd_list_price', 'num_colors', 'has_prime', 
            'num_images', 'sd_aplus', 'title_word_count', 'sharpness_laplacian_var_mean']:
    if col in df.columns:
        non_null = df[col].notna().sum()
        print(f"  {col}: {non_null} non-null ({non_null/len(df)*100:.1f}%)")

Loading dataset...
Loaded 36879 products, 7771 columns
  price: 36387 non-null (98.7%)
  sd_price: 35384 non-null (95.9%)
  sd_list_price: 35489 non-null (96.2%)
  num_colors: 36879 non-null (100.0%)
  has_prime: 36879 non-null (100.0%)
  num_images: 36879 non-null (100.0%)
  sd_aplus: 36741 non-null (99.6%)
  title_word_count: 36879 non-null (100.0%)
  sharpness_laplacian_var_mean: 36879 non-null (100.0%)


## 1. Price Features

In [2]:
# Use sd_price if available (more reliable), fall back to price
# NOTE: price is a pre-listing feature (seller sets price before listing)
# sd_price and sd_list_price are the seller's chosen prices
price_col = None
for candidate in ['sd_price', 'price']:
    if candidate in df.columns:
        price_col = candidate
        break

if price_col:
    # Convert to numeric, handle non-numeric gracefully
    df['_price_numeric'] = pd.to_numeric(df[price_col], errors='coerce')
    
    # Log price (handle 0 and NaN)
    df['log_price'] = np.log1p(df['_price_numeric'].fillna(0).clip(lower=0))
    
    # Price bins per category (budget/mid/premium)
    # NOTE: category medians are computed at training time and frozen for inference
    # This is NOT look-ahead - at inference we use stored medians from training data
    cat_col = 'main_bsr_group' if 'main_bsr_group' in df.columns else 'category'
    
    def categorize_price(row, medians):
        price = row['_price_numeric']
        cat = row.get(cat_col)
        if pd.isna(price) or pd.isna(cat) or cat not in medians:
            return 1  # mid as default
        median = medians[cat]
        if price < median * 0.5:
            return 0  # budget
        elif price > median * 1.5:
            return 2  # premium
        return 1  # mid
    
    # Compute per-category price medians
    category_price_medians = df.groupby(cat_col)['_price_numeric'].median().to_dict()
    df['price_bin'] = df.apply(lambda r: categorize_price(r, category_price_medians), axis=1)
    
    # Price vs category median
    df['price_vs_category_median'] = df.apply(
        lambda r: (r['_price_numeric'] - category_price_medians.get(r.get(cat_col), r['_price_numeric'])) / 
                  max(category_price_medians.get(r.get(cat_col), 1), 0.01)
        if pd.notna(r['_price_numeric']) else 0,
        axis=1
    )
    
    # REMOVED: has_discount and discount_pct
    # Discount status is a post-listing promotional decision that can change over time
    # and may correlate with BSR as an effect (discounted products sell more) rather than cause
    
    # Clean up temp column
    df.drop('_price_numeric', axis=1, inplace=True)
    
    print(f"Price features created using '{price_col}' column")
    for feat in ['log_price', 'price_bin', 'price_vs_category_median']:
        if feat in df.columns:
            print(f"  {feat}: mean={df[feat].mean():.3f}, std={df[feat].std():.3f}")
else:
    print("No price column found - skipping price features")
    df['log_price'] = 0
    df['price_bin'] = 1
    df['price_vs_category_median'] = 0

Price features created using 'sd_price' column
  log_price: mean=3.354, std=1.165
  price_bin: mean=1.088, std=0.564
  price_vs_category_median: mean=0.750, std=19.251


## 2. Product Completeness & Interaction Features

In [3]:
# Product completeness features
# All features here are PRE-LISTING (seller decisions made before/at listing time)
num_images_col = 'num_images' if 'num_images' in df.columns else None
if num_images_col:
    df['has_multiple_images'] = (df[num_images_col] > 1).astype(int)
else:
    df['has_multiple_images'] = 0

# Number of colors (seller-defined product variants)
if 'num_colors' in df.columns:
    df['num_colors_available'] = pd.to_numeric(df['num_colors'], errors='coerce').fillna(0)
else:
    df['num_colors_available'] = 0

# A+ content flag (seller creates A+ content before/at listing time)
if 'sd_aplus' in df.columns:
    df['has_aplus_content'] = df['sd_aplus'].fillna(False).astype(int)
else:
    df['has_aplus_content'] = 0

# Interaction features (all inputs are pre-listing)
# Title length x sharpness (text quality x image quality interaction)
sharpness_col = None
for candidate in ['sharpness_laplacian_var_mean', 'sharpness_mean', 'sharpness']:
    if candidate in df.columns:
        sharpness_col = candidate
        break

title_wc = df.get('title_word_count', pd.Series([0]*len(df)))

if sharpness_col:
    # Normalize both to 0-1 range before multiplying
    sharpness_norm = df[sharpness_col].fillna(0) / max(df[sharpness_col].max(), 1)
    title_norm = title_wc / max(title_wc.max(), 1)
    df['title_length_x_sharpness'] = title_norm * sharpness_norm
else:
    df['title_length_x_sharpness'] = 0

# Price x image count (both seller-controlled pre-listing)
if num_images_col:
    df['price_x_image_count'] = df['log_price'] * df[num_images_col].fillna(1)
else:
    df['price_x_image_count'] = df['log_price']

# Text richness x image count (title words + bullet words) * images
bullets_wc = df.get('bullets_total_word_count', pd.Series([0]*len(df)))
text_richness = title_wc + bullets_wc
if num_images_col:
    df['text_richness_x_image_count'] = text_richness * df[num_images_col].fillna(1)
else:
    df['text_richness_x_image_count'] = text_richness

# Category-relative title length
# NOTE: category medians computed at training time, frozen for inference (not look-ahead)
cat_col = 'main_bsr_group' if 'main_bsr_group' in df.columns else 'category'
category_title_medians = df.groupby(cat_col)['title_word_count'].median().to_dict() if 'title_word_count' in df.columns else {}
df['title_length_vs_category_median'] = df.apply(
    lambda r: (r.get('title_word_count', 0) - category_title_medians.get(r.get(cat_col), 0)) / 
              max(category_title_medians.get(r.get(cat_col), 1), 1)
    if pd.notna(r.get('title_word_count')) else 0,
    axis=1
)

# Summary
eng_features = [
    'log_price', 'price_bin', 'price_vs_category_median',
    'has_multiple_images', 'num_colors_available', 'has_aplus_content',
    'title_length_x_sharpness', 'price_x_image_count', 
    'text_richness_x_image_count', 'title_length_vs_category_median',
]

print(f"Created {len(eng_features)} engineered features:")
for feat in eng_features:
    if feat in df.columns:
        print(f"  {feat}: mean={df[feat].mean():.3f}, std={df[feat].std():.3f}")

Created 10 engineered features:
  log_price: mean=3.354, std=1.165
  price_bin: mean=1.088, std=0.564
  price_vs_category_median: mean=0.750, std=19.251
  has_multiple_images: mean=0.000, std=0.000
  num_colors_available: mean=0.825, std=1.950
  has_aplus_content: mean=0.958, std=0.200
  title_length_x_sharpness: mean=0.021, std=0.022
  price_x_image_count: mean=3.354, std=1.165
  text_richness_x_image_count: mean=22.871, std=7.612
  title_length_vs_category_median: mean=6.105, std=10.284


## 3. Save Final Enriched Dataset

In [4]:
# Full list of ALL new features added across notebooks 03-05
all_new_features = []

# TF-IDF SVD features (from 03_tfidf_features)
tfidf_feats = [f'title_tfidf_pca_{i:04d}' for i in range(50)] + [f'bullets_tfidf_pca_{i:04d}' for i in range(50)]
all_new_features.extend([f for f in tfidf_feats if f in df.columns])

# NLP text stats (from 04_advanced_nlp)
nlp_feats = [
    'title_char_count', 'title_word_count', 'title_avg_word_length',
    'title_unique_word_ratio', 'title_flesch_reading_ease', 'title_flesch_kincaid_grade',
    'title_separator_count', 'title_has_brand', 'title_has_size_spec', 'title_has_color_spec',
    'bullets_count', 'bullets_avg_length', 'bullets_total_word_count',
    'has_bullets', 'bullets_keyword_density',
]
all_new_features.extend([f for f in nlp_feats if f in df.columns])

# Engineered features (from this notebook)
# NOTE: has_discount and discount_pct were REMOVED - they are post-listing promotional decisions
eng_feats = [
    'log_price', 'price_bin', 'price_vs_category_median',
    'has_multiple_images', 'num_colors_available', 'has_aplus_content',
    'title_length_x_sharpness', 'price_x_image_count',
    'text_richness_x_image_count', 'title_length_vs_category_median',
]
all_new_features.extend([f for f in eng_feats if f in df.columns])

print(f"Total new features across all NLP notebooks: {len(all_new_features)}")
print(f"  TF-IDF SVD: {len([f for f in all_new_features if 'tfidf_pca' in f])}")
print(f"  NLP text stats: {len([f for f in all_new_features if f in nlp_feats])}")
print(f"  Engineered: {len([f for f in all_new_features if f in eng_feats])}")
print(f"\nFinal DataFrame shape: {df.shape}")

# Save
OUTPUT_CSV = "../../data/products_with_image_feats.csv"
print(f"\nSaving to {OUTPUT_CSV}...")
df.to_csv(OUTPUT_CSV, index=False)
print("Done! All feature engineering complete.")
print(f"\nNext: Run save_models_for_web.py to retrain models with these features.")

Total new features across all NLP notebooks: 125
  TF-IDF SVD: 100
  NLP text stats: 15
  Engineered: 10

Final DataFrame shape: (36879, 7781)

Saving to ../../data/products_with_image_feats.csv...
Done! All feature engineering complete.

Next: Run save_models_for_web.py to retrain models with these features.
